# Fine Res Initial

In [41]:
import os
python_path = r"C:\Users\QT3 User Facility\AppData\Local\Programs\Python\Python314\Scripts"
os.environ["PATH"] = python_path + ";" + os.environ["PATH"]

# Test
!pip --version

pip 25.3 from C:\Users\QT3 User Facility\AppData\Local\Programs\Python\Python314\Lib\site-packages\pip (python 3.14)



In [42]:
# (requires) setup.py, run to show version of qickdawg installed
!pip show qickdawg

Name: qickdawg
Version: 1.2.1
Summary: Software for full quantum control of nitrogen-vacancy defects and other quantum defects in diamond
Home-page: 
Author: 
Author-email: Andy Mounce <amounce@sandia.gov>, Emmeline Riendeau <eriendeau@uchicago.edu>
License: MIT License 

Copyright 2023 National Technology & Engineering Solutions of Sandia, LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the U.S. Government retains certain rights in this software.

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substa

In [97]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from copy import copy
import qickdawg as qd

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

from importlib.metadata import version
version('numpy')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'2.3.3'

In [98]:
qd.start_client('192.168.3.1')

In [ ]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.mw_channel = 0
default_config.freq_fMHz = 500
default_config.mw_gain = 5000 # check user if do two amps for min max
default_config.mw_nqz = 1

# Sweep params
# should check the user to make sure min samples is something and max
# Triggering
default_config.laser_gate_pmod = 0 
default_config.laser_init_tus = 2
default_config.readout_integration_tns = 500
default_config.mw_to_laser_delay_tns = 555
default_config.laser_readout_offset_tus = 1.2

default_config.pulse_seq_delay_tus = 1
default_config.pre_init = True

In [108]:
from qickdawg.nvtestsuite.rabi_fine_res import RabiFineRes

config = copy(default_config)

config.add_unitless_linear_sweep("mw_duration_tdds", 5000, 5005, delta=5) # the sweep is inclusive of the start and end values
config.reps = 1
config.freq_fMHz = 500 # 200

config.pulse_seq_delay_tus = 0.2

prog = RabiFineRes(config)
d=prog.acquire(progress = True)



Requested 5000 to 5005 by 5
Instead using 5000 to 5010 by 5 in 2


100%|██████████| 2/2 [00:00<00:00, 259.77it/s]


In [116]:
from qickdawg.nvtestsuite.counting_duration_fine_res import CountingDurationFineRes

config = copy(default_config)

config.mw_duration_tdds = 5000
config.freq_fMHz = 500 # 200
config.reps = 100000

config.pulse_seq_delay_tus = 0.2

# Sweeping "laser_readout_offset_treg"
laser_readout_offset_step_tns = 100


for i in range(20):

    config.laser_readout_offset_treg = i * laser_readout_offset_step_treg

    prog = CountingDurationFineRes(config)

    d = prog.acquire()

    if i ==0:
        results = d
        results.delay = np.array([config.laser_readout_offset_treg])
    else:
        for key in d.keys():
            results[key] = np.append(results[key], d[key])
        results.delay = np.append(results.delay, [config.laser_readout_offset_treg])



In [117]:
config = copy(default_config)

config.mw_nqz # 1 at 1405 MHz
config.mw_fMHz = 200
config.mw_gain = 2000
config.reps = 2

config.mw_start_tsamp = 2000
config.mw_end_tsamp = 2002
config.nsweep_points = 2

# Laser params
config.laser_init_tus = 1

# Readout and delays
config.readout_integration_tns = 300
config.mw_to_laser_delay_tns = 505
config.laser_readout_offset_tus = 1.2

prog = qd.RABI_SUBNANO(config)
d = prog.acquire(progress=True)

AssertionError: Missing value for config.laser_gate_pmod